# Can we predict next-month arrears?
Protocol: `protocol.yaml` (temporal holdout from `holdout.csv`, 5-split walk-forward CV on training months, choose by mean CV ROC AUC, score holdout once).

Feature decisions (from `columns.csv`):
- `meter_ref` (entity key) and `billing_month` (split column) are excluded per protocol.
- `final_notice_sent` is **excluded**: it is recorded in the billing cycle *after* the billing month, i.e. after prediction time, and is almost a proxy for the outcome (86.5% arrears when 1 vs 1.4% when 0). It would leak the outcome.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='42accdc3f8c6ae3514896d73516d98c54e4a27c5422d4fe88e5c4fe1b00a14f7'
df = pd.read_csv('data.csv', parse_dates=['billing_month'])
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
is_hold = np.zeros(len(df), bool); is_hold[hold_idx] = True
train, hold = df[~is_hold], df[is_hold]
print(len(train), len(hold), train.billing_month.max(), hold.billing_month.min())
assert train.billing_month.max() < hold.billing_month.min()

In [ ]:
TARGET='arrears_next_month'
CAT=['tariff']; NUM=['occupants','kwh','bill_eur','late_payments_12m']
FEATURES=CAT+NUM
def pre(): return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),('n',StandardScaler(),NUM)])
# All candidates named before any is scored
candidates = {
 'baseline_prior': Pipeline([('p',pre()),('m',DummyClassifier(strategy='prior'))]),
 'logreg': Pipeline([('p',pre()),('m',LogisticRegression(max_iter=1000))]),
 'random_forest': Pipeline([('p',pre()),('m',RandomForestClassifier(n_estimators=300,min_samples_leaf=20,n_jobs=1,random_state=0))]),
 'hist_gb': Pipeline([('p',pre()),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

Walk-forward CV over whole billing months within the training rows (expanding window, 5 splits).

In [ ]:
months = np.sort(train.billing_month.unique())
folds=[]
for tr_m, te_m in TimeSeriesSplit(n_splits=5).split(months):
    tr = train.index[train.billing_month.isin(months[tr_m])].to_numpy()
    te = train.index[train.billing_month.isin(months[te_m])].to_numpy()
    folds.append((tr,te))
    print(str(months[tr_m][0])[:7], str(months[tr_m][-1])[:7], '->', str(months[te_m][0])[:7], str(months[te_m][-1])[:7], len(tr), len(te))
cv_scores={}
for name,m in candidates.items():
    s=[]
    for tr,te in folds:
        f=clone(m).fit(df.loc[tr,FEATURES], df.loc[tr,TARGET])   # preprocessing fitted inside the fold
        s.append(float(roc_auc_score(df.loc[te,TARGET], f.predict_proba(df.loc[te,FEATURES])[:,1])))
    cv_scores[name]=s
cv = pd.DataFrame(cv_scores).T; cv['mean']=cv.mean(axis=1); cv

In [ ]:
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
print('chosen by mean CV roc_auc:', chosen)
final = clone(candidates[chosen]).fit(train[FEATURES], train[TARGET])
hold_auc = float(roc_auc_score(hold[TARGET], final.predict_proba(hold[FEATURES])[:,1]))
print('HOLDOUT roc_auc (scored once):', round(hold_auc,4))

In [ ]:
evidence = dict(
  features=FEATURES, candidates=list(candidates),
  cv_folds=[[tr.tolist(), te.tolist()] for tr,te in folds],
  cv_scores=cv_scores, chosen=chosen,
  holdout_rows=hold.index.tolist(), holdout_scores={chosen: hold_auc})
json.dump(evidence, open('evidence.json','w'))

## Conclusion
See the CV table and holdout ROC AUC above. The holdout score is the headline; it comes from the model chosen on CV only. `final_notice_sent` was deliberately not used, so this is an at-prediction-time estimate.